# Adversarial Vulnerability of Positional Encoding in ViTs — Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djokobandjur/vit-positional-adversarial/blob/main/colab_quickstart.ipynb)

This notebook reproduces the corrected attack pipeline for the paper *"Adversarial Vulnerability of Positional Encoding in Vision Transformers"*. It demonstrates the corrected multi-block attack and all post-hoc analyses (E2: ALiBi structural ablation; E3: attention reorganization and saturation; noise ablation for the decoupling thesis).

**Layout:**
- Repository scripts are cloned fresh into `/content/vit-positional-adversarial/`.
- Trained models live on Google Drive (large, persistent).
- ImageNet-100 val tarball lives on Google Drive; the setup script extracts it into `/content/imagenet100/val/`.
- CIFAR-100 is auto-downloaded by torchvision into `/content/drive/MyDrive/cifar100_cache/`.
- All generated JSON results are written to `/content/drive/MyDrive/patching_framework/data/`.
- Figures are written to `/content/drive/MyDrive/patching_framework/figures/`.

Each section can be run independently provided trained models and prior JSON outputs are available.

## 1. Setup

Mount Google Drive (for trained models, datasets, and persistent JSON output), verify GPU availability, and clone the repository into `/content/`.

### 1.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Verify GPU

All scripts require a CUDA-capable GPU. The full pipeline was developed and tested on G4 GPU.

In [ ]:
!nvidia-smi

### 1.3 Clone repository

Scripts are pulled fresh from GitHub on every session.

In [ ]:
%cd /content
!git clone https://github.com/djokobandjur/vit-positional-adversarial.git
%cd /content/vit-positional-adversarial

### 1.4 Create output directories on Drive

All JSON results and figures land in the consolidated `data/` and `figures/` folders on Drive, mirroring the local layout of the repo.

In [ ]:
import os

DATA_DIR = '/content/drive/MyDrive/patching_framework/data'
FIG_DIR  = '/content/drive/MyDrive/patching_framework/figures'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('Output dirs ready:')
print(' ', DATA_DIR)
print(' ', FIG_DIR)

## 2. Dataset Preparation

ImageNet-100 requires a one-time setup step (extracting 100 classes from the ILSVRC2012 val tarball). CIFAR-100 is downloaded automatically by torchvision on first use, so no preparation cell is needed for it.

### 2.1 ImageNet-100 setup

`00_setup_imagenet.py` reads the ILSVRC2012 val tar archive from Drive and unpacks the 100 classes used in this work (50 images per class) into `/content/imagenet100/val/`. The archive itself is **not** unpacked manually — the script handles extraction internally.

Expected input on Drive: `/content/drive/MyDrive/pe_experiment/imagenet/ILSVRC2012_img_val.tar`

In [ ]:
!python /content/vit-positional-adversarial/00_setup_imagenet.py

### 2.2 CIFAR-100

No preparation needed: the attack and analysis scripts call `torchvision.datasets.CIFAR100(download=True)` internally. The dataset cache is stored under `/content/drive/MyDrive/cifar100_cache/` so re-downloads are avoided across sessions.

## 3. Adversarial Attacks (the corrected attack)

`full_reanalysis.py` is the corrected attack pipeline. It replaces the previous `adversarial_pe_attacks.py` (which contained the multi-block bug) and the deprecated attack section of `cifar100_experiment.py`. The same script handles both datasets via `--dataset {imagenet,cifar}`.

Output per run: nested JSON `{pe_type: {seed: {attack: {eps: acc}}}}` for FGSM-PE, PGD-PE, and VTA across 8 ε values and 3 seeds. Results are checkpointed after each `(pe_type, seed)` combination so interrupted sessions can resume.

### 3.1 ImageNet-100 attacks

In [ ]:
!python /content/vit-positional-adversarial/full_reanalysis.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/patching_framework/data/imagenet_results.json" \
    --dataset imagenet

### 3.2 CIFAR-100 attacks

In [ ]:
!python /content/vit-positional-adversarial/full_reanalysis.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --val_dir "/content/drive/MyDrive/cifar100_cache" \
    --output_path "/content/drive/MyDrive/patching_framework/data/cifar_results.json" \
    --dataset cifar

## 4. Experiment 2: ALiBi Structural Ablation

Decomposes the ALiBi mechanism into its two components (per-head slopes and relative-distance matrix) and attacks each in isolation. Establishes that the slopes carry essentially all of the structural vulnerability — perturbing only the relative-distance term leaves accuracy nearly unchanged, while perturbing only the slopes recovers the joint attack effect.

Output: `{pe_type: {seed: {variant: {eps: acc}}}}` where `variant ∈ {slopes_only, reldist_only, both}`.

### 4.1 ImageNet-100

In [ ]:
!python /content/vit-positional-adversarial/experiment2_alibi_ablation.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/patching_framework/data/imagenet_alibi_ablation.json" \
    --dataset imagenet

### 4.2 CIFAR-100

In [ ]:
!python /content/vit-positional-adversarial/experiment2_alibi_ablation.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --val_dir "/content/drive/MyDrive/cifar100_cache" \
    --output_path "/content/drive/MyDrive/patching_framework/data/cifar_alibi_ablation.json" \
    --dataset cifar

## 5. Experiment 3: Attention Reorganization and Saturation

Two complementary measurements on the attacked models:

- **Spatial metrics** (concentration ratio, MAD): how attention distributions reorganize under the attack — whether the attack induces sharper focus, broader diffusion, or no spatial restructuring.
- **Perturbation norms** (`frac@ceil`): how often the per-step perturbation update hits its allowed L∞ ceiling during PGD. High saturation indicates the attacker is bounded by the budget rather than by gradient signal.

### 5.1 Spatial attention metrics — ImageNet-100

In [ ]:
!python /content/vit-positional-adversarial/experiment3_attention_metrics_v3.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/patching_framework/data/imagenet_spatial_metrics.json" \
    --dataset imagenet \
    --batch_size 128

### 5.2 Spatial attention metrics — CIFAR-100

CIFAR-100 runs do not require `--val_dir` (the script invokes torchvision's auto-download internally).

In [ ]:
!python /content/vit-positional-adversarial/experiment3_attention_metrics_v3.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --output_path "/content/drive/MyDrive/patching_framework/data/cifar_spatial_metrics.json" \
    --dataset cifar \
    --batch_size 128

### 5.3 Perturbation norms / saturation — ImageNet-100

In [ ]:
!python /content/vit-positional-adversarial/experiment3_perturbation_norm_v4.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/patching_framework/data/imagenet_perturbation_norms.json" \
    --dataset imagenet \
    --batch_size 128

### 5.4 Perturbation norms / saturation — CIFAR-100

In [ ]:
!python /content/vit-positional-adversarial/experiment3_perturbation_norm_v4.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --output_path "/content/drive/MyDrive/patching_framework/data/cifar_perturbation_norms.json" \
    --dataset cifar \
    --batch_size 128

## 6. Noise Ablation (Decoupling Thesis, ImageNet-100 only)

Measures clean-model accuracy under Gaussian noise added independently to each transformer block's PE buffer, at 8 noise levels. This populates the left panel of Figure 2 in the paper and underpins the **decoupling thesis** — that the random-noise robustness ranking (Learned ≫ Sinusoidal > RoPE > ALiBi) is fundamentally different from the adversarial ranking (RoPE ≫ Learned ≈ Sinusoidal ≫ ALiBi).

This script is ImageNet-100 only; no CIFAR equivalent is provided in this toolkit.

In [ ]:
!python /content/vit-positional-adversarial/extract_tables_data.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/patching_framework/data/analysis_data.json"

## 7. Generate Figures

Consumes all 9 JSON files produced above and emits 12 figures (7 main paper + 5 supplement) to the `figures/` directory. Pass `--format png` for raster output or `--format both` to emit both PDF and PNG.

In [ ]:
!python /content/vit-positional-adversarial/generate_figures.py \
    --imagenet         /content/drive/MyDrive/patching_framework/data/imagenet_results.json \
    --cifar            /content/drive/MyDrive/patching_framework/data/cifar_results.json \
    --imagenet-ablation /content/drive/MyDrive/patching_framework/data/imagenet_alibi_ablation.json \
    --cifar-ablation    /content/drive/MyDrive/patching_framework/data/cifar_alibi_ablation.json \
    --imagenet-spatial  /content/drive/MyDrive/patching_framework/data/imagenet_spatial_metrics.json \
    --cifar-spatial     /content/drive/MyDrive/patching_framework/data/cifar_spatial_metrics.json \
    --imagenet-norms    /content/drive/MyDrive/patching_framework/data/imagenet_perturbation_norms.json \
    --cifar-norms       /content/drive/MyDrive/patching_framework/data/cifar_perturbation_norms.json \
    --outdir            /content/drive/MyDrive/patching_framework/figures/ \
    --format pdf

## Done

All artifacts are now on Drive:

- **9 JSON result files** in `/content/drive/MyDrive/patching_framework/data/`
- **12 PDF figures** in `/content/drive/MyDrive/patching_framework/figures/` (7 main paper + 5 supplement)

For details on the corrected multi-block attack and the decoupling vs. inversion thesis, see the `README.md` and `CHANGELOG.md` in the repository.